# Combined Algorithm Selection and Hyperparameter optimization (CASH)

Notebook for the course [Master Hyperparameter Optimization for Tabular Learning](http://www.trainindata.com/p/master-hyperparameter-optimization-for-tabular-learning)

<div style="
    padding: 12px 16px;
    border-left: 5px solid #2196f3;
    background-color: #eaf4fd;
    border-radius: 4px;
">
<strong>Note:</strong>
Combined Algorithm Selection and Hyperparameter Optimization (CASH) refers to the process of simultaneously choosing a machine learning algorithm and tuning its hyperparameters to improve predictive performance. 
</div>

The aim is: given one total computational budget, find the best deployable configuration, regardless of model family.

Useful framework for:
- Automated ML systems where model identity is just another hyperparameter.
- Large numbers of candidate algorithms and conditional preprocessing steps.
- Repeated modeling across many datasets, customers, products, or regions.
- A strict shared compute or time budget.
- Situations where you want the optimizer to stop spending trials on consistently weak model families.

In this notebook, we will use the define-by-run framework to optimize the hyperparameters of various machine learning models from Scikit-learn.

In [1]:
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

import optuna

In [2]:
# load dataset

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
y = y.map({0:1, 1:0})

X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# the target:
# percentage of benign (0) and malignant tumors (1)

y.value_counts() / len(y)

target
0    0.627417
1    0.372583
Name: count, dtype: float64

In [4]:
# split dataset into a train and test set

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

X_train.shape, X_test.shape

((398, 30), (171, 30))

## Define the objective function

This is the hyperparameter response space, the function we want to optimize.

In [5]:
# the objective function takes the hyperparameter space
# as input

def objective(trial):
    
    classifier_name = trial.suggest_categorical("classifier", ["logit", "RF", 'GBM'])
    
    if classifier_name == "logit":

        model = LogisticRegression(
            l1_ratio = trial.suggest_float('logit_l1ratio', 0, 1),
            C = trial.suggest_float('logit_c', 0.001, 10),
            solver = 'saga',
            max_iter = 10000,
        )
        
    elif classifier_name =="RF":

        model = RandomForestClassifier(
            n_estimators = trial.suggest_int("rf_n_estimators", 100, 1000),
            criterion = trial.suggest_categorical("rf_criterion", ['gini', 'entropy']),
            max_depth = trial.suggest_int("rf_max_depth", 1, 4),
            min_samples_split = trial.suggest_float("rf_min_samples_split", 0.01, 1),
        )
        
    else:

        model = GradientBoostingClassifier(
            n_estimators = trial.suggest_int("gbm_n_estimators", 100, 1000),
            max_depth = trial.suggest_int("gbm_max_depth", 1, 4),
            min_samples_split = trial.suggest_float("gbm_min_samples_split", 0.01, 1),
            learning_rate = trial.suggest_float("gbm_learning_rate", 0.01, 10),
        )

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
    scores = []

    for step, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):

        fold_model = clone(model)
        fold_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])

        predictions = fold_model.predict_proba(X_train.iloc[valid_idx])[:, 1]
        score = roc_auc_score(y_train.iloc[valid_idx], predictions)
        scores.append(score)

        trial.report(np.mean(scores), step)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)

## Random search with median pruning

<div style="padding: 12px 16px; border-left: 5px solid #2196f3; background-color: #eaf4fd; border-radius: 4px;">
<strong>Note:</strong> Based on its benchmarks for non-deep-learning tasks, Optuna recommends combining <code>RandomSampler</code> with <code>MedianPruner</code>. See Optuna's <a href="https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/003_efficient_optimization_algorithms.html#which-sampler-and-pruner-should-be-used">sampler and pruner recommendations</a>.
</div>

In [6]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(seed=0),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1,
    ),
)

study.optimize(objective, n_trials=20)

[I 2026-08-26 10:59:30,659] A new study created in memory with name: no-name-7dae6861-8d0f-4704-b74a-ac93592d412f
[I 2026-08-26 10:59:32,942] Trial 0 finished with value: 0.7161847389558234 and parameters: {'classifier': 'GBM', 'gbm_n_estimators': 964, 'gbm_max_depth': 3, 'gbm_min_samples_split': 0.162985341000892, 'gbm_learning_rate': 6.169259949612765}. Best is trial 0 with value: 0.7161847389558234.
[I 2026-08-26 10:59:33,555] Trial 1 finished with value: 0.9872649782804688 and parameters: {'classifier': 'RF', 'rf_n_estimators': 483, 'rf_criterion': 'entropy', 'rf_max_depth': 2, 'rf_min_samples_split': 0.030697344635905174}. Best is trial 1 with value: 0.9872649782804688.
[I 2026-08-26 10:59:34,515] Trial 2 finished with value: 0.985730677813294 and parameters: {'classifier': 'RF', 'rf_n_estimators': 778, 'rf_criterion': 'entropy', 'rf_max_depth': 2, 'rf_min_samples_split': 0.2497076660610464}. Best is trial 1 with value: 0.9872649782804688.
[I 2026-08-26 10:59:34,689] Trial 3 finis

In [7]:
study.best_params

{'classifier': 'GBM',
 'gbm_n_estimators': 534,
 'gbm_max_depth': 1,
 'gbm_min_samples_split': 0.013553992704340156,
 'gbm_learning_rate': 0.11455949592013503}

In [8]:
study.best_value

0.9924301286779773

In [9]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_classifier,params_gbm_learning_rate,params_gbm_max_depth,params_gbm_min_samples_split,params_gbm_n_estimators,params_logit_c,params_logit_l1ratio,params_rf_criterion,params_rf_max_depth,params_rf_min_samples_split,params_rf_n_estimators,state
0,0,0.716185,2026-08-26 10:59:30.659505,2026-08-26 10:59:32.942267,0 days 00:00:02.282762,GBM,6.169260,3.0,0.162985,964.0,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.987265,2026-08-26 10:59:32.942647,2026-08-26 10:59:33.555403,0 days 00:00:00.612756,RF,NaN,NaN,NaN,NaN,NaN,NaN,entropy,2.0,0.030697,483.0,COMPLETE
2,2,0.985731,2026-08-26 10:59:33.555728,2026-08-26 10:59:34.515648,0 days 00:00:00.959920,RF,NaN,NaN,NaN,NaN,NaN,NaN,entropy,2.0,0.249708,778.0,COMPLETE
3,3,0.373962,2026-08-26 10:59:34.515950,2026-08-26 10:59:34.689489,0 days 00:00:00.173539,GBM,4.252252,3.0,0.679926,156.0,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.992081,2026-08-26 10:59:34.689784,2026-08-26 10:59:35.567996,0 days 00:00:00.878212,GBM,0.223496,2.0,0.364938,864.0,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
5,5,0.500000,2026-08-26 10:59:35.568260,2026-08-26 10:59:36.150188,0 days 00:00:00.581928,RF,NaN,NaN,NaN,NaN,NaN,NaN,entropy,4.0,0.722053,556.0,COMPLETE
6,6,0.986949,2026-08-26 10:59:36.150521,2026-08-26 10:59:36.971452,0 days 00:00:00.820931,RF,NaN,NaN,NaN,NaN,NaN,NaN,entropy,4.0,0.126736,606.0,COMPLETE
7,7,0.416737,2026-08-26 10:59:36.971779,2026-08-26 10:59:38.042740,0 days 00:00:01.070961,GBM,4.138320,2.0,0.801070,997.0,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
8,8,0.963946,2026-08-26 10:59:38.043055,2026-08-26 10:59:38.469059,0 days 00:00:00.426004,logit,NaN,NaN,NaN,NaN,5.746474,0.197673,NaN,NaN,NaN,NaN,COMPLETE
9,9,0.500000,2026-08-26 10:59:38.469328,2026-08-26 10:59:38.732406,0 days 00:00:00.263078,RF,NaN,NaN,NaN,NaN,NaN,NaN,gini,4.0,0.911812,246.0,COMPLETE


In [10]:
results = study.trials_dataframe()

results['params_classifier'].value_counts()

params_classifier
GBM      12
RF        5
logit     3
Name: count, dtype: int64

Random search samples the classifier independently in each trial. Differences in the number of trials per classifier are therefore due to random variation, not adaptive preference for one model.

In [11]:
results.groupby(['params_classifier'])['value'].agg(['mean', 'std'])

,mean,std
params_classifier,,
GBM,0.846731,0.224709
RF,0.791989,0.266549
logit,0.963975,0.000050


## How median pruning works in this example

Each cross-validation fold is treated as one pruning step. The value reported to Optuna is the cumulative mean ROC AUC:

| Step | Value reported |
|:---:|:---|
| 0 | ROC AUC from fold 1 |
| 1 | Mean ROC AUC from folds 1 and 2 |
| 2 | Mean ROC AUC from folds 1, 2, and 3 |

With `n_startup_trials=5`, the first five completed trials establish the reference results and are not pruned. With `n_warmup_steps=1`, subsequent trials are also protected at step 0. At steps 1 and 2, Optuna compares the current trial's best intermediate result so far with the median intermediate result reported at the same step by previous completed trials. Because this study maximizes ROC AUC, a result below the reference median may be pruned; an equal or higher result continues.

For example, if the first five completed trials report `[0.76, 0.81, 0.79, 0.74, 0.83]` at step 1, the reference median is `0.79`. If a later trial reports a cumulative mean ROC AUC of `0.75`, `trial.should_prune()` may return `True`, and its third fold is not fitted.

> **Note:** Pruning here saves complete fold fits; it does not interrupt a model while that fold is being fitted. With only three folds and the first step protected, pruning can avoid only the final fold fit. With 5- or 10-fold cross-validation, an early decision can avoid more remaining fits, making the potential savings larger. However, every trial that is not pruned also becomes more expensive, and reliable pruning still requires enough completed trials and comparable fold ordering.


![Median pruning with cross-validation](images/median-pruner-cross-validation.png)